# Rotary Positional Embeddings (RoPE): How It Works and Why It Beats Learned Embeddings

> This notebook contains all code examples from the blog post.
> [Read the full post on BotMartz](https://blog.botmartz.com/research_2)

**Author:** Soham Sharma · blog.botmartz.com

In [ ]:
!pip install -q torch transformers peft bitsandbytes

In [ ]:
import torch
import math

def precompute_rope_freqs(head_dim: int, max_seq_len: int, base: float = 10000.0) -> tuple:
    """
    Precompute the cosine and sine frequencies for RoPE.
    Returns: (cos, sin) each of shape (max_seq_len, head_dim // 2)
    """
    # θ_i = 1 / (base^(2i/d)) for i in [0, d/2)
    i = torch.arange(0, head_dim, 2, dtype=torch.float32)
    thetas = 1.0 / (base ** (i / head_dim))  # shape: (head_dim // 2,)

    # Positions 0, 1, 2, ..., max_seq_len - 1
    positions = torch.arange(max_seq_len, dtype=torch.float32)  # (max_seq_len,)

    # Outer product: freqs[m, i] = m * theta_i
    freqs = torch.outer(positions, thetas)  # (max_seq_len, head_dim // 2)

    return freqs.cos(), freqs.sin()

cos_freqs, sin_freqs = precompute_rope_freqs(head_dim=64, max_seq_len=8)
print(f"cos_freqs shape: {cos_freqs.shape}")
print(f"First 4 theta values: {1.0 / (10000 ** (torch.arange(0, 8, 2).float() / 64))[:4]}")
print(f"Frequencies range: min={cos_freqs.min():.4f}, max={cos_freqs.max():.4f}")

In [ ]:
import torch

def rotate_half(x: torch.Tensor) -> torch.Tensor:
    """
    Rotate by 90 degrees: swaps and negates alternate dimensions.
    x shape: (..., head_dim)
    """
    x1 = x[..., : x.shape[-1] // 2]   # first half
    x2 = x[..., x.shape[-1] // 2 :]   # second half
    return torch.cat([-x2, x1], dim=-1)

def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """
    Apply RoPE to query or key tensor.
    x:   (batch, heads, seq_len, head_dim)
    cos: (seq_len, head_dim // 2)  → broadcast to (1, 1, seq_len, head_dim)
    sin: same
    """
    # Repeat cos/sin to match full head_dim (each angle used twice)
    cos = cos.repeat_interleave(2, dim=-1)  # (seq_len, head_dim)
    sin = sin.repeat_interleave(2, dim=-1)

    # Broadcast to (1, 1, seq_len, head_dim)
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]

    # Rotation: x * cos + rotate_half(x) * sin
    return x * cos + rotate_half(x) * sin

# Test: verify that dot product is position-relative
torch.manual_seed(42)
B, H, S, D = 1, 1, 4, 8

cos_f, sin_f = precompute_rope_freqs(head_dim=D, max_seq_len=S)

q = torch.randn(B, H, S, D)
k = torch.randn(B, H, S, D)

q_rot = apply_rope(q, cos_f, sin_f)
k_rot = apply_rope(k, cos_f, sin_f)

# Attention scores before and after RoPE
scores_no_rope = torch.einsum('bhid,bhjd->bhij', q, k)
scores_rope    = torch.einsum('bhid,bhjd->bhij', q_rot, k_rot)

print("Scores without RoPE:")
print(scores_no_rope[0, 0].round(decimals=3))
print("\nScores with RoPE (relative position encoded):")
print(scores_rope[0, 0].round(decimals=3))

In [ ]:
import torch
import math

def precompute_rope_freqs(head_dim, max_seq_len, base=10000.0):
    i = torch.arange(0, head_dim, 2, dtype=torch.float32)
    thetas = 1.0 / (base ** (i / head_dim))
    positions = torch.arange(max_seq_len, dtype=torch.float32)
    freqs = torch.outer(positions, thetas)
    return freqs.cos(), freqs.sin()

def rotate_half(x):
    x1, x2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]
    return torch.cat([-x2, x1], dim=-1)

def apply_rope(x, cos, sin):
    cos = cos.repeat_interleave(2, dim=-1)[None, None]
    sin = sin.repeat_interleave(2, dim=-1)[None, None]
    return x * cos + rotate_half(x) * sin

torch.manual_seed(7)
D = 16
cos_f, sin_f = precompute_rope_freqs(D, 10)

q = torch.randn(1, 1, 1, D)  # single query vector
k = torch.randn(1, 1, 1, D)  # single key vector

results = {}
for m in range(5):
    for n in range(5):
        cos_m, sin_m = cos_f[m:m+1], sin_f[m:m+1]
        cos_n, sin_n = cos_f[n:n+1], sin_f[n:n+1]
        q_rot = apply_rope(q, cos_m, sin_m)
        k_rot = apply_rope(k, cos_n, sin_n)
        score = (q_rot * k_rot).sum().item()
        rel = m - n
        if rel not in results:
            results[rel] = []
        results[rel].append(round(score, 5))

print("Relative position → dot product scores (should be same value per row):")
for rel in sorted(results.keys()):
    vals = results[rel]
    consistent = len(set(vals)) == 1
    print(f"  rel={rel:+d}: {vals[0]:.5f} × {len(vals)} {'✓' if consistent else '✗'}")

In [ ]:
import torch
import math

def precompute_rope_freqs_yarn(head_dim: int, max_seq_len: int,
                                base: float = 10000.0,
                                scale_factor: float = 1.0) -> tuple:
    """
    YaRN-extended RoPE: scale the base to extend context window.
    scale_factor = new_context / original_context
    """
    # Scaled base: higher base = slower frequency decay = longer range
    scaled_base = base * (scale_factor ** (head_dim / (head_dim - 2)))

    i = torch.arange(0, head_dim, 2, dtype=torch.float32)
    thetas = 1.0 / (scaled_base ** (i / head_dim))
    positions = torch.arange(max_seq_len, dtype=torch.float32)
    freqs = torch.outer(positions, thetas)
    return freqs.cos(), freqs.sin()

# Original: trained at 4096 tokens
cos_orig, _ = precompute_rope_freqs_yarn(64, 4096, scale_factor=1.0)
# YaRN 2×: extends to 8192 tokens
cos_yarn2x, _ = precompute_rope_freqs_yarn(64, 8192, scale_factor=2.0)

print(f"Original theta[0] (highest freq): {1.0 / (10000 ** 0):.6f}")
scaled_base_2x = 10000.0 * (2.0 ** (64 / 62))
print(f"YaRN 2× theta[0] (highest freq): {1.0 / (scaled_base_2x ** 0):.6f}")
print(f"Effect: slows rotation, enables longer range")

In [ ]:
from transformers import AutoConfig

# Load the config only (no weights)
config = AutoConfig.from_pretrained("meta-llama/Llama-3.2-1B")

print(f"Max position embeddings: {config.max_position_embeddings}")
print(f"RoPE base: {config.rope_theta}")
print(f"Head dim: {config.hidden_size // config.num_attention_heads}")
print(f"Rope scaling: {getattr(config, 'rope_scaling', 'None')}")